In [ ]:
###Import Packages###
import os
import pandas as pd
import pyjstat
import json
from pyjstat import pyjstat
import requests


from modules.functions import *




: 

In [2]:
MIN_YEAR = 2012

In [ ]:
#------------------------------------------------------------Constants----------------------------------------------------------
BARKPERCENTAGE = 0.10
ROTFACTOR = 1/15
MORTALITYFACTOR = 0.15
#------------------------------------------------------------Local data----------------------------------------------------------
path_excel_data = './data/excel_data' 

treindustrien_df = pd.read_csv(f'{path_excel_data}/treindustrien production data.csv')
fao_boardmills_df = pd.read_csv(f'{path_excel_data}/FAO_Data.csv')

#------------------------------------------------------------SSB Imports----------------------------------------------------------
ssb_api_output =  process_query_files('./data/SSB_queries')

#------------------------------------------------------------SSB TABLE 03795----------------------------------------------------------
tbl_03795 = pd.concat(ssb_api_output['03795'],ignore_index=True)
tbl_03795['sortiment'] = tbl_03795['sortiment'].replace({
    'Gran':'Spruce',
    'Furu':'Pine',
    'Lauv':'Non-coniferous'
})

tbl_03795 = tbl_03795[tbl_03795['år'].astype(int)>=MIN_YEAR]
tbl_03795 = tbl_03795[tbl_03795['sortiment'] != 'Ved til brensel']
rows_03795 = pd.DataFrame({'source':'Standing Wood',
                         'target':tbl_03795['sortiment'].astype(str),
                         'volume':tbl_03795['value'].astype(float),
                         'year':tbl_03795['år'].astype(int),
                         'origin':'SSB table 03795'
                         })
#------------------------------------------------------------SSB TABLE 11181----------------------------------------------------------
tbl_11181 = pd.concat(ssb_api_output['11181'],ignore_index=True)
tbl_11181['value']=tbl_11181['value']*1000 #Transform the data from (1000m3) to (1m3)
firewood_coniferous = tbl_11181[tbl_11181['virkesgruppe']!='Ved til brensel lauv']

non_coniferous_firewood_df = tbl_11181[tbl_11181['virkesgruppe']=='Ved til brensel lauv']

def firewood_ratio(percent_spruce):
    spruce = float(percent_spruce/100)
    pine = float((1-spruce))

    return spruce, pine


percent_spruce = 50
spruce_ratio,pine_ratio = firewood_ratio(percent_spruce=percent_spruce)

spruce_firewood_df = firewood_coniferous.assign(
    virkesgruppe = 'Spruce',
    value = firewood_coniferous['value']*spruce_ratio
)
rows_firewood_spruce =  pd.DataFrame({'source':'Spruce',
                            'target':'Firewood',
                            'volume':spruce_firewood_df['value'].astype(float),
                            'year':spruce_firewood_df['år'].astype(int),
                            'origin':'SSB table 11181'
                         })      
rows_firewood_spruce_sw= pd.DataFrame({
                            'source':'Standing wood',
                            'target':'Spruce',
                            'volume':spruce_firewood_df['value'].astype(float),
                            'year':spruce_firewood_df['år'].astype(int),
                            'origin':'SSB table 11181'
                         })



pine_firewood_df = firewood_coniferous.assign(
    virkesgruppe = 'Pine',
    value = firewood_coniferous['value']*pine_ratio
)

rows_firewood_pine =  pd.DataFrame({'source':'Pine',
                         'target':'Firewood',
                         'volume':pine_firewood_df['value'].astype(float),
                         'year':pine_firewood_df['år'].astype(int),
                         'origin':'SSB table 11181'
                         })

rows_firewood_pine_sw= pd.DataFrame({
                            'source':'Standing wood',
                            'target':'Pine',
                            'volume':pine_firewood_df['value'].astype(float),
                            'year':pine_firewood_df['år'].astype(int),
                            'origin':'SSB table 11181'
                         })

rows_non_coniferous_firewood = pd.DataFrame({'source':'Non-coniferous',
                                             'target':'Firewood',
                                             'volume':non_coniferous_firewood_df['value'],
                                             'year':non_coniferous_firewood_df['år'],
                                             'origin':'SSB table 11181'})

rows_non_coniferous_firewood_sw= pd.DataFrame({
                            'source':'Standing wood',
                            'target':'Non-coniferous',
                            'volume':non_coniferous_firewood_df['value'].astype(float),
                            'year':non_coniferous_firewood_df['år'].astype(int),
                            'origin':'SSB table 11181'
                         })

tbl_06289 = pd.concat(ssb_api_output['06289'],ignore_index=True)
tilvekst_df = tbl_06289.loc[
    (tbl_06289['statistikkvariabel'] == 'Årlig tilvekst') & (tbl_06289['treslag'] == 'I alt')
].copy()

#------------------------------------------------------------SSB TABLE 10513----------------------------------------------------------
tbl_10513 = pd.concat(ssb_api_output['10513'],ignore_index=True)
tbl_10513 = tbl_10513[tbl_10513['value']!=0]
tbl_10513['volume'] = tbl_10513['value']*1000/483 # tonnes -> kg -> m3
tbl_10513['source'] = 'Long-term USE'


tbl_10513.loc[tbl_10513['behandlingsmåte'] == 'Levert til materialgjenvinning','target'] = 'Material recovery'
tbl_10513.loc[tbl_10513['behandlingsmåte'] == 'Levert til forbrenning','target'] = 'Energy'
tbl_10513.loc[~tbl_10513['behandlingsmåte'].isin(['Levert til materialgjenvinning','Levert til forbrenning']) ,'target'] = 'Other'

tbl_10513_sum = (
    tbl_10513
    .groupby("år", as_index=False)["volume"]
    .sum()
)

rows_EOL = pd.DataFrame(
    {
        'year':tbl_10513_sum['år'],
        'volume':tbl_10513_sum['volume'],
        'source':'Long-term USE',
        'target':'End-of-Life',
        'origin': 'SSB table 10513'
    }
)

rows_waste = pd.DataFrame(
    {
        'year': tbl_10513['år'],
        'volume': tbl_10513['volume'],
        'source': 'End-of-Life',
        'target': tbl_10513['target'],
        'origin':'SSB table 10513'
    }
)


# Use .loc to assign a new column using an explicit column label indexer
val_idx = tilvekst_df.columns.get_loc('value')
tilvekst_df.loc[:, 'mortality'] = tilvekst_df.iloc[:, val_idx] * 1000 * MORTALITYFACTOR

rows_mortality = pd.DataFrame({'source':'Standing wood',
                                             'target':'Mortality',
                                             'volume':tilvekst_df['mortality'],
                                             'year':tilvekst_df['år'],
                                             'origin':'SSB table 06289'})




#------------------------------------------------------------SSB TABLE 07410----------------------------------------------------------
tbl_07410 = pd.concat(ssb_api_output['07410'],ignore_index=True)
tbl_07410['value'] = tbl_07410['value']*1000 #1000m3 -> 1m3

tbl_07410_sum = (
    tbl_07410
    .groupby(["år"], as_index=False)["value"]
    .sum()
)

tbl_07410_sum_sans_lauv = (
    tbl_07410
    .groupby(["år","sortiment"], as_index=False)["value"]
    .sum()
)
tbl_07410_sum_sans_lauv = tbl_07410_sum_sans_lauv[~tbl_07410_sum_sans_lauv['sortiment'].isin([
    'Lauvtre massevirke', 
    'Lauvtre sagtømmer',
    'Lauvtre sams sagtømmer og massevirke'
])]
tbl_07410_sum_sans_lauv= tbl_07410_sum_sans_lauv.groupby('år', as_index=False).sum(numeric_only=True)


rows_bark = pd.DataFrame(
    {
        'year':tbl_07410_sum_sans_lauv['år'],
        'volume':tbl_07410_sum_sans_lauv['value']*0.10,
        'source':'Standing wood',
        'target':'Bark',
        'origin': 'SSB table 07410'
    }
)

rows_residues = pd.DataFrame(
    {
        'year':tbl_07410_sum['år'],
        'volume':tbl_07410_sum['value']*0.10,
        'source':'Standing wood',
        'target':'Residues from harvest',
        'origin': 'SSB table 07410'
    }
)

df_gran_sag = tbl_07410[tbl_07410['sortiment']== 'Gran sagtømmer']
df_gran_masse = tbl_07410[tbl_07410['sortiment']== 'Gran massevirke']
df_gran_board = tbl_07410[tbl_07410['sortiment']== 'Gran sams sagtømmer og massevirke']

df_furu_sag = tbl_07410[tbl_07410['sortiment']== 'Furu sagtømmer']
df_furu_masse = tbl_07410[tbl_07410['sortiment']== 'Furu massevirke']
df_furu_board = tbl_07410[tbl_07410['sortiment']== 'Furu sams sagtømmer og massevirke']

df_lauv_sag = tbl_07410[tbl_07410['sortiment']== 'Lauvtre sagtømmer']
df_lauv_masse = tbl_07410[tbl_07410['sortiment']== 'Lauvtre massevirke']
df_lauv_board = tbl_07410[tbl_07410['sortiment']== 'Lauvtre sams sagtømmer og massevirke']

#------------------------------------------------------------SSB TABLE 08801----------------------------------------------------------


#------------------------------------------------------------GRAN----------------------------------------------------------
rows_gran_sag = pd.DataFrame({'source':'Spruce',
                                             'target':'Sawmills',
                                             'volume':df_gran_sag['value'],
                                             'year':df_gran_sag['år'],
                                             'origin':'SSB table 07410'})
rows_gran_masse = pd.DataFrame({'source':'Spruce',
                                             'target':'Pulpmills',
                                             'volume':df_gran_masse['value'],
                                             'year':df_gran_masse['år'],
                                             'origin':'SSB table 07410'})
rows_gran_board = pd.DataFrame({'source':'Spruce',
                                             'target':'Boardmills',
                                             'volume':df_gran_board['value'],
                                             'year':df_gran_board['år'],
                                             'origin':'SSB table 07410'})
rows_gran = pd.concat ([rows_gran_sag,rows_gran_masse,rows_gran_board],ignore_index=True)
rows_gran = rows_gran.groupby('year', as_index=False)['volume'].sum()

rows_gran = pd.DataFrame({'source':'Standing wood',
                                             'target':'Spruce',
                                             'volume':rows_gran['volume'],
                                             'year':rows_gran['year'],
                                             'origin':'SSB table 07410'})
#------------------------------------------------------------FURU----------------------------------------------------------
rows_furu_sag = pd.DataFrame({'source':'Pine',
                                             'target':'Sawmills',
                                             'volume':df_furu_sag['value'],
                                             'year':df_furu_sag['år'],
                                             'origin':'SSB table 07410'})
rows_furu_masse = pd.DataFrame({'source':'Pine',
                                             'target':'Pulpmills',
                                             'volume':df_furu_masse['value'],
                                             'year':df_furu_masse['år'],
                                             'origin':'SSB table 07410'})
rows_furu_board = pd.DataFrame({'source':'Pine',
                                             'target':'Boardmills',
                                             'volume':df_furu_board['value'],
                                             'year':df_furu_board['år'],
                                             'origin':'SSB table 07410'})
rows_furu = pd.concat ([rows_furu_sag,rows_furu_masse,rows_furu_board],ignore_index=True)
rows_furu = rows_furu.groupby('year', as_index=False)['volume'].sum()

rows_furu = pd.DataFrame({'source':'Standing wood',
                                             'target':'Pine',
                                             'volume':rows_furu['volume'],
                                             'year':rows_furu['year'],
                                             'origin':'SSB table 07410'})
#------------------------------------------------------------LAUV----------------------------------------------------------
rows_lauv_sag = pd.DataFrame({'source':'Non-coniferous',
                                             'target':'Sawmills',
                                             'volume':df_lauv_sag['value'],
                                             'year':df_lauv_sag['år'],
                                             'origin':'SSB table 07410'})
rows_lauv_masse = pd.DataFrame({'source':'Non-coniferous',
                                             'target':'Pulpmills',
                                             'volume':df_lauv_masse['value'],
                                             'year':df_lauv_masse['år'],
                                             'origin':'SSB table 07410'})
rows_lauv_board = pd.DataFrame({'source':'Non-coniferous',
                                             'target':'Boardmills',
                                             'volume':df_lauv_board['value'],
                                             'year':df_lauv_board['år'],
                                             'origin':'SSB table 07410'})

rows_lauv = pd.concat ([rows_lauv_sag,rows_lauv_masse,rows_lauv_board],ignore_index=True)
rows_lauv = rows_lauv.groupby('year', as_index=False)['volume'].sum()

rows_lauv = pd.DataFrame({'source':'Standing wood',
                                             'target':'Non-coniferous',
                                             'volume':rows_lauv['volume'],
                                             'year':rows_lauv['year'],
                                             'origin':'SSB table 07410'})

tbl_08801 = pd.concat(ssb_api_output['08801'],ignore_index=True)
tbl_08801 = tbl_08801.loc[tbl_08801['value']!=0]
tbl_08801['type'] = 'default'

tbl_08801.loc[tbl_08801['varenummer'].str.contains('Impregnert',case=False),'type'] = 'impregnated'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('Massevirke',case=False),'type'] = 'pulpwood'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('Skurtømmer',case=False),'type'] = 'saw wood'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('Trepeletter',case=False),'type'] = 'pellets'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('brensel',case=False),'type'] = 'firewood'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('høvellast',case=False),'type'] = 'planed wood'
tbl_08801.loc[tbl_08801['varenummer'].str.contains('skurlast',case=False),'type'] = 'sawn wood'

tbl_08801.loc[tbl_08801['statistikkvariabel'] == 'Mengde 1 (M1)', 'statistikkvariabel'] = 'kg'
tbl_08801.loc[tbl_08801['statistikkvariabel'] == 'Mengde 2 (M2)', 'statistikkvariabel'] = 'm3'

trade_df = assign_target_source_trade(tbl_08801)
trade_df = trade_df.rename(columns={'år': 'year'})
trade_df_volume = trade_df.copy()
trade_df_volume = trade_df_volume[trade_df_volume['statistikkvariabel']=='m3']
trade_df_volume =trade_df_volume.groupby(['year','source','target'], as_index=False)['value'].sum()
trade_df_volume = trade_df_volume.rename(columns={'value':'volume'})






Processed file: ssbapi_table_03795.json, Table ID: 03795
Processed file: ssbapi_table_06289.json, Table ID: 06289
Processed file: ssbapi_table_07410.json, Table ID: 07410
Processed file: ssbapi_table_08801.json, Table ID: 08801
Processed file: ssbapi_table_10513.json, Table ID: 10513
Processed file: ssbapi_table_11181.json, Table ID: 11181


In [ ]:
#Write SSB data to sankey_df
sankey_df = pd.DataFrame(columns=['source','target','volume','year','origin'])
sankey_df = pd.concat([sankey_df,rows_03795],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_waste],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_firewood_spruce],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_firewood_pine],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_non_coniferous_firewood],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_gran_sag],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_gran_masse],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_gran_board],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_furu_sag],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_furu_masse],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_furu_board],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_lauv_sag],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_lauv_masse],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_lauv_board],ignore_index=True)


#------------------------------------------------------------Treindustrien Data----------------------------------------------------------

df_planed = (
    treindustrien_df[treindustrien_df["Category"] == "produced planed timber"]
    .drop(columns="Category")
    .rename(columns={"Volume": "Volume_planed"})
)
df_sawn = (
    treindustrien_df[treindustrien_df["Category"] == "produced sawn timber"]
    .drop(columns="Category")
    .rename(columns={"Volume": "Volume_sawn"})
)
df_imp = (
    treindustrien_df[treindustrien_df["Category"] == "produsert impregnated timber"]
    .drop(columns="Category")
    .rename(columns={"Volume": "Volume_imp"})
)

df_merged = df_sawn.merge(df_planed, on="year").merge(df_imp, on="year")
print("Merged DataFrame:\n", df_merged, "\n")


flows_df =create_flow_rows(
    sankey_df=sankey_df,
    treindustrien_production=df_merged,
    treindustrien_inventory=treindustrien_df,
    trade_df=trade_df_volume)



Merged DataFrame:
     year  Volume_sawn  Volume_planed   Volume_imp
0   2011  2209249.343    1272627.020  375581.9680
1   2012  2241078.174    1341313.400  364821.2800
2   2013  2216166.627    1288677.600  368555.9200
3   2014  2407879.512    1308106.100  412745.4968
4   2015  2444016.256    1323120.960  431269.1440
5   2016  2532248.386    1394301.720  413416.3640
6   2017  2655347.836    1403662.540  381079.9600
7   2018  2675121.922    1366384.600  371070.0000
8   2019  2662015.300    1325843.120  411346.9600
9   2020  2689750.179    1268159.200  463639.5200
10  2021  2808777.193    1417321.840  435952.7200
11  2022  2701628.018    1183785.208  308199.4560
12  2023  2489504.625    1146707.613  327370.0000 



c:\Users\mla\circwood\Circwood_WP4\Volume_dataset\modules\functions.py:322: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inventory_df.loc[:, 'VolumeChange'] = inventory_df.groupby('Category')[


In [12]:
flows_df

,year,volume,source,target,origin
0,2011,1.336258e+06,Sawmills,Planingmills,Treindustrien årsrapport
1,2011,8.729910e+05,Sawmills,Wood Product processing & Logistics,Treindustrien årsrapport
2,2011,1.803000e+05,Sawmills,Shrinkage,Treindustrien årsrapport
3,2011,4.795980e+05,Sawmills,Boardmills,Treindustrien årsrapport
4,2011,8.906820e+05,Sawmills,Pulpmills,Treindustrien årsrapport
...,...,...,...,...,...
125,2023,2.866034e+05,Sawmills,Boardmills,Treindustrien årsrapport
126,2023,6.020215e+04,Planingmills,Boardmills,Treindustrien årsrapport
127,2023,8.193376e+05,Planingmills,Wood Product processing & Logistics,Treindustrien årsrapport
128,2023,3.273700e+05,Planingmills,Impregnation,Treindustrien årsrapport


In [7]:
def create_treindustrien_inventory_rows(treindustrien_df):
    inventory_mapping = {
        'Timber inventory': {
            'positive': ('Sawmills', 'Timber Inventory'),
            'negative': ('Timber Inventory', 'Sawmills')
        },
        'sawn timber inventory': {
            'positive': ('Sawmills', 'Timber Inventory'),
            'negative': ('Timber Inventory', 'Sawmills')
        },
        'planed timber inventory': {
            'positive': ('Planingmills', 'planed timber inventory'),
            'negative': ('planed timber inventory', 'Planingmills')
        },
        'Impregnated inventory': {
            'positive': ('Impregnation', 'Impregnated inventory'),
            'negative': ('Impregnated inventory', 'Impregnation')
        }
    }

    treindustrien_df['source'], treindustrien_df['target'] = zip(
        *treindustrien_df.apply(get_source_target_treindustrien, axis=1))
    treindustrien_df_inventory = treindustrien_df[treindustrien_df['Category'].isin(
        list(inventory_mapping.keys()))].reset_index().reset_index(drop=True)
    treindustrien_df_inventory = treindustrien_df_inventory.dropna()

    treindustrien_rows_inventory = pd.DataFrame({

        'year': treindustrien_df_inventory['year'],
        'volume': treindustrien_df_inventory['VolumeChange'],
        'source': treindustrien_df_inventory['source'],
        'target': treindustrien_df_inventory['target'],
        'origin': 'Treindustrien årsrapport'
    })

    return treindustrien_rows_inventory

In [8]:
treindustrien_df.sort_values(['Category','year'])
treindustrien_df['VolumeChange'] = treindustrien_df.groupby('Category')['Volume'].diff().abs()


treindustrien_df_inventory = create_treindustrien_inventory_rows(treindustrien_df)

In [9]:
treindustrien_df_inventory

,year,volume,source,target,origin
1,2012,1164.7797,Sawmills,Timber Inventory,Treindustrien årsrapport
2,2013,115769.8033,Sawmills,Timber Inventory,Treindustrien årsrapport
3,2014,213595.7998,Sawmills,Timber Inventory,Treindustrien årsrapport
4,2015,165750.4169,Sawmills,Timber Inventory,Treindustrien årsrapport
5,2016,60144.0498,Sawmills,Timber Inventory,Treindustrien årsrapport
6,2017,85107.8446,Sawmills,Timber Inventory,Treindustrien årsrapport
7,2018,16390.3821,Sawmills,Timber Inventory,Treindustrien årsrapport
8,2019,124216.3484,Sawmills,Timber Inventory,Treindustrien årsrapport
9,2020,176028.2625,Sawmills,Timber Inventory,Treindustrien årsrapport
10,2021,241046.8840,Sawmills,Timber Inventory,Treindustrien årsrapport


In [10]:
treindustrien_df

,year,Category,Volume,VolumeChange,source,target
0,2011,produced sawn timber,2209249.343,NaN,None,None
1,2012,produced sawn timber,2241078.174,31828.831,None,None
2,2013,produced sawn timber,2216166.627,24911.547,None,None
3,2014,produced sawn timber,2407879.512,191712.885,None,None
4,2015,produced sawn timber,2444016.256,36136.744,None,None
...,...,...,...,...,...,...
86,2019,Impregnated inventory,117990.000,33354.880,Impregnation,Impregnated inventory
87,2020,Impregnated inventory,68071.680,49918.320,Impregnation,Impregnated inventory
88,2021,Impregnated inventory,124246.320,56174.640,Impregnation,Impregnated inventory
89,2022,Impregnated inventory,108576.640,15669.680,Impregnation,Impregnated inventory


In [ ]:



sankey_df = pd.concat([sankey_df,trade_df_volume],ignore_index=True)
sankey_df = pd.concat([sankey_df,flows_df],ignore_index=True)
sankey_df = pd.concat([sankey_df,treindustrien_df_inventory],ignore_index=True)
sankey_df = pd.concat([sankey_df,fao_boardmills_df],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_EOL],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_bark],ignore_index=True)
sankey_df = pd.concat ([sankey_df,rows_residues],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_gran,rows_furu,rows_lauv],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_firewood_spruce_sw,rows_firewood_pine_sw,rows_non_coniferous_firewood_sw],ignore_index=True)
sankey_df = pd.concat([sankey_df,rows_mortality],ignore_index=True)


sankey_df = boardmill_process(dataframe=sankey_df)
sankey_df = pulpmills_process(dataframe=sankey_df,
                short_term_products=86,
                  biofuel=8,
                  loss=6)

df_firewood = (
    sankey_df[sankey_df['target'] == 'Firewood']
    .groupby('year', as_index=False)['volume']
    .sum()
)


df_firewood['source'] ='Firewood'
df_firewood['target'] ='Energy'
df_firewood['origin'] ='SSB various tables'

sankey_df = pd.concat([sankey_df,df_firewood],ignore_index=True)

df_wpp = (
    sankey_df[sankey_df['target'] == 'Wood Product processing & Logistics']
    .groupby('year', as_index=False)['volume']
    .sum()
)

df_wpp['source'] = 'Wood Product processing & Logistics'
df_wpp['origin'] = 'SSB and Treindustrien'


df_wpp_ltu = df_wpp.copy()
df_wpp_ltu['volume'] = df_wpp_ltu['volume'] * 0.90
df_wpp_ltu['target'] = 'Long-term USE'

df_wpp_energy = df_wpp.copy()
df_wpp_energy['volume'] = df_wpp_energy['volume'] * 0.10
df_wpp_energy['target'] = 'Energy'

sankey_df = pd.concat([sankey_df, df_wpp_ltu, df_wpp_energy], ignore_index=True)

sankey_df['year']=pd.to_numeric(sankey_df['year'],errors='coerce')
sankey_df['year']=sankey_df['year'].astype(int)
sankey_df['volume']=sankey_df['volume'].abs()
sankey_df.to_clipboard()

In [18]:
sankey_df_grouped = sankey_df.groupby(['source','target','year','origin'], as_index=False)['volume'].sum()
sankey_df_grouped.to_clipboard()

In [15]:
sankey_df.to_clipboard()

In [ ]:
sankey_df[sankey_df['target'] == 'Sawmills'].groupby(
        "year", as_index=False)["volume"].sum()

In [ ]:
intial_timber_vol = sankey_df[sankey_df['target'] == 'Sawmills'].groupby(
        "year", as_index=False)["volume"].sum() +treindustrien_df[treindustrien_df['category']
                                        == 'Timber inventory'].groupby(
                                            'year', as_index=False)['Volume'].sum()

In [ ]:
def density_calculator(species, moisture_content):
    mc_percent = moisture_content/100
    if species == "spruce":
        avg_dry_density = 430
        FSP = 32/100
        max_volume_shrinkage = 11.7/100
        
    elif species == "pine":
        avg_dry_density = 490
        FSP = 27/100
        max_volume_shrinkage = 12.1/100
    elif species == "birch":
        avg_dry_density = 580
        FSP = 34/100
        max_volume_shrinkage = 17.4/100
    
    
    Density_at_given_mc = (
        100-mc_percent*max_volume_shrinkage/FSP)*(1+mc_percent)/100*avg_dry_density
    dry_mass_at_given_moisture_content = Density_at_given_mc/(1+mc_percent)
    carbon_content = dry_mass_at_given_moisture_content/2
    
    return Density_at_given_mc,carbon_content
    
    
    #Strategy for density (%spruce,%pine,%lauv)

print(density_calculator('spruce',15))




In [ ]:
carbon_df = sankey_df.copy().rename(columns={'volume':'carbon'})

In [ ]:
carbon_df['carbon'] = carbon_df['carbon']*density_calculator(species='spruce',moisture_content=100)[1]


In [ ]:
sankey_df.to_csv('sankey_data_from_python.csv')

In [ ]:
tbl_07410 = pd.concat(ssb_api_output['07410'],ignore_index=True)
tbl_07410['value'] = tbl_07410['value']*1000 #1000m3 -> 1m3

tbl_07410_sum = (
    tbl_07410
    .groupby(["år"], as_index=False)["value"]
    .sum()
)

tbl_07410_sum_sans_lauv = (
    tbl_07410
    .groupby(["år","sortiment"], as_index=False)["value"]
    .sum()
)
tbl_07410_sum_sans_lauv = tbl_07410_sum_sans_lauv[~tbl_07410_sum_sans_lauv['sortiment'].isin([
    'Lauvtre massevirke', 
    'Lauvtre sagtømmer',
    'Lauvtre sams sagtømmer og massevirke'
])]
tbl_07410_sum_sans_lauv= tbl_07410_sum_sans_lauv.groupby('år', as_index=False).sum(numeric_only=True)


rows_bark = pd.DataFrame(
    {
        'year':tbl_07410_sum_sans_lauv['år'],
        'volume':tbl_07410_sum_sans_lauv['value']*0.10,
        'source':'Standing wood',
        'target':'Bark',
        'origin': 'SSB table 07410'
    }
)

rows_residues = pd.DataFrame(
    {
        'year':tbl_07410_sum['år'],
        'volume':tbl_07410_sum['value']*0.10,
        'source':'Standing wood',
        'target':'Residues from harvest',
        'origin': 'SSB table 07410'
    }
)

In [ ]:
sawmills_df = sankey_df[sankey_df['target'] == 'Sawmills']

sawmills_df = sawmills_df[sawmills_df['source'].isin(['Spruce','Pine'])]
sawmills_df.to_clipboard()

In [ ]:
standing_wood_df = sankey_df[sankey_df['source']=='Standing wood']
standing_wood_df.to_clipboard()

In [ ]:
sankey_df[sankey_df['target']=='Sawmills'].to_clipboard()

In [ ]:
firewood_df = sankey_df[sankey_df['target']=='Firewood']
firewood_df.to_clipboard()

In [ ]:
pulpmills_df = sankey_df[sankey_df['target']=='Pulpmills']
pulpmills_df.to_clipboard()

In [ ]:
energy_df = sankey_df[sankey_df['target']=='Energy']
energy_df.to_clipboard()

In [ ]:
intial_timber_vol = sankey_df[sankey_df['source']=='Sawmills'].groupby("year", as_index=False)["volume"].sum()
intial_timber_vol

In [ ]:
tbl_11181 = pd.concat(ssb_api_output['11181'], ignore_index=True)

# Suppose we want 60% Spruce, 40% Pine
firewood_df = create_firewood_rows(tbl_11181, percent_spruce=50)

print(firewood_df.head())

In [ ]:
def assign_source_target(row, species):
    """
    Assign a (source, target) tuple based on the species and the product keywords
    found in 'varenummer'. This version ignores net exports and simply applies
    a single source-target mapping for each match.

    Parameters:
    -----------
    row : pandas.Series
        Expects at least a 'varenummer' column.
    species : str
        The species classification (e.g. 'gran', 'furu', 'bjørk', etc.) to determine
        which dictionary to use.

    Returns:
    --------
    (str, str)
        A (source, target) pair. If no match is found, returns (None, None).
    """
    varenummer_lower = row['varenummer'].lower()

    # Example mappings (adjust as needed for your data and process flow)
    if species in ['gran', 'furu']:
        product_map = {
            'massevirke': ('Forest', 'Pulpmills'),
            'skurtømmer': ('Forest', 'Sawmills'),
            'tømmer':     ('Forest', 'Boardmills'),
            'impregnert': ('Impregnation', 'Impregnated Inventory'),
            'skurlast':   ('Sawmills', 'Sawnwood Inventory'),
            'høvellast':  ('Planingmills', 'Planedwood Inventory')
        }
    elif species in ['lauv', 'bjørk', 'brensel']:
        product_map = {
            'brensel': ('Forest', 'Firewood Processing'),
            'lauv':    ('Forest', 'Firewood Processing'),
            'bjørk':   ('Forest', 'Firewood Processing')
        }
    elif species == 'bartre':
        product_map = {
            'impregnert': ('Impregnation', 'Impregnated Inventory'),
            'skurtømmer': ('Forest', 'Sawmills'),
            'uimpregnert': ('Sawmills', 'Planingmills'),
            'massevirke': ('Forest', 'Pulpmills')
        }
    else:
        # If species doesn't match any known groups
        return (None, None)

    # Search for the product keyword within varenummer
    for product_keyword, (source, target) in product_map.items():
        if product_keyword in varenummer_lower:
            return (source, target)

    # If nothing matched
    return (None, None)


In [ ]:
varekoder_df = pd.DataFrame(tbl_08801['varenummer'].unique().tolist())
varekoder_df.to_json('varekoder.json',encoding ='iso-8859-1')

In [ ]:
def sawmills_inventory_changes(treindustrien_df):

    treindustrien_df.sort_values(['Category','year'])
    inventory_df = treindustrien_df[treindustrien_df['Category'].str.contains('inventory',case=False,na=False)]
    inventory_df.loc[:,'VolumeChange'] = inventory_df.groupby('Category')['Volume'].diff()

    
    return inventory_df

inventory_df = sawmills_inventory_changes(treindustrien_df=treindustrien_df)

inventory_df

NameError: name 'ssb_api_output' is not defined

In [ ]:
trade_df_vol = trade_df.loc[trade_df['statistikkvariabel']=='m3']
trade_df_vol.head(5)

,varenummer,import/eksport,land,statistikkvariabel,year,value,type,source,target
111,"Ved til brensel (1988-2016) (M1=kg, M2=m³)",Import,Belarus,m3,2003,28,firewood,Import Firewood,Firewood
112,"Ved til brensel (1988-2016) (M1=kg, M2=m³)",Import,Belarus,m3,2004,101,firewood,Import Firewood,Firewood
158,"Ved til brensel (1988-2016) (M1=kg, M2=m³)",Import,Belgia,m3,2006,10,firewood,Import Firewood,Firewood
207,"Ved til brensel (1988-2016) (M1=kg, M2=m³)",Import,Bosnia-Hercegovina,m3,2011,10,firewood,Import Firewood,Firewood
212,"Ved til brensel (1988-2016) (M1=kg, M2=m³)",Import,Bosnia-Hercegovina,m3,2016,6,firewood,Import Firewood,Firewood


In [ ]:

trade_df_vol = trade_df.loc[trade_df['statistikkvariabel']=='m3']
trade_df_rows = make_flow(df=trade_df_vol,
                          volume_col = 'value',
                          year_col= 'year',
                          source_col='source',
                          target_col='target',
                          origin='SSB Table 08801'
                          )

trade_df_rows['source'].unique()

array(['Import Firewood', 'Firewood', 'Import saw wood', 'Sawmills',
       'Import pulpwood', 'Pulpmills', 'Import sawn wood',
       'Import planed wood', 'Planingmills'], dtype=object)

In [ ]:
import_saw_wood = trade_df_vol[trade_df_vol['source']=='Import saw wood'].groupby(
    "year", as_index=False)['value'].sum()
import_saw_wood

,year,value
0,2002,295737
1,2003,497769
2,2004,577391
3,2005,582571
4,2006,241280
5,2007,288378
6,2008,199171
7,2009,237229
8,2010,311141
9,2011,168039
